In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _mesh_data(graph):
    from topologicpy.TGraph import TGraph
    data = TGraph.MeshData(graph)
    data["vertexDictionaries"] = [v["dictionary"] for v in TGraph.Vertices(graph)]
    data["edgeDictionaries"] = [e["dictionary"] for e in _edge_records(graph)]
    return data


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


## Graph Booleans

In [ ]:
# Import TopologicPy modules. This is not needed on other computers
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

### Import the needed classes

In [ ]:
from topologicpy.Edge import Edge
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.Plotly import Plotly
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color
print("TopologicPy Classes imported.")

### Create two example graphs with some overlaps (vertices that have the same "id" key/value)
### The "id" is visualised as the size of the vertex

In [ ]:
# Step 1: Base graph
topology = CellComplex.Prism()
g1 = TGraph.ByTopology(topology, toExteriorTopologies=True)

# Give each vertex in the graph a unique sequential id and a prurple color.
verts1 = TGraph.Vertices(g1)
for i, v in enumerate(verts1):
    d = Dictionary.ByKeysValues(["id", "color"], [str(i+5), "purple"])
    v = Topology.SetDictionary(v, d)

# Create a second graph by moving the vertices and edges to a new location.
verts2 = [Topology.Translate(_as_topology(v), 3, 0, 1) for v in verts1]
e1 = _edge_records(g1)
edges2 = [Topology.Translate(_as_topology(e), 3, 0, 1) for e in e1]

# Remove 15 random vertices (and edges) from g1
for i in range(15):
    g1 = TGraph.RemoveVertex(g1, verts1[0])

g1 = TGraph.RemoveIsolatedVertices(g1)
verts1 = TGraph.Vertices(g1)
for vert in verts1:
    d = _record_dictionary(vert)
    d = Dictionary.SetValueAtKey(d, "color", "orange") # Second graph has orange nodes and one additional red node.
    vert = Topology.SetDictionary(vert, d)
e1 = _edge_records(g1)


# Get the mesh data of g1 to reconstruct it later on.
mesh_data = _mesh_data(g1)
edges1 = mesh_data['edges']
vert_dicts1 = mesh_data['vertexDictionaries']
edge_dicts1 = mesh_data['edgeDictionaries']


g2 = TGraph.ByVerticesEdges(verts2, edges2)
verts2 = TGraph.Vertices(g2)
edges2 = _edge_records(g2)


# Add parent key to vertex and edge dictionaries:
vertices_g1 = TGraph.Vertices(g1)
edges_g1 = _edge_records(g1)
both = vertices_g1+edges_g1
for b in both:
    d = _record_dictionary(b)
    d = Dictionary.SetValueAtKey(d, "parent", "g1")
    b = Topology.SetDictionary(b, d)

vertices_g2 = TGraph.Vertices(g2)
edges_g2 = _edge_records(g2)
both = vertices_g2+edges_g2
for b in both:
    d = _record_dictionary(b)
    d = Dictionary.SetValueAtKey(d, "parent", "g2")
    b = Topology.SetDictionary(b, d)

# Add an extra random edge to g1 (red node)
vertices = TGraph.Vertices(g1)
v1 = vertices[0]
v2 = Topology.Translate(_as_topology(v1), 0.4, -0.2, 0.1)
d = _record_dictionary(v1)
d = Dictionary.SetValuesAtKeys(d, ["parent","id", "color"], ["g1",len(vertices_g2), "red"]) # This is the additional red node.
v2 = Topology.SetDictionary(v2, d)
e = Edge.ByVertices(_as_topology(v1), _as_topology(v2))
d = Dictionary.ByKeysValues(['category', 'parent', 'relationship'], [100, 'g1', 'Undefined'])
e = Topology.SetDictionary(e, d)
TGraph.AddEdge(g1, e, transferEdgeDictionaries= True)

# Step 3: Show the two graphs
p1 = Plotly.DataByGraph(g1, edgeWidth=2, vertexSizeKey="id", vertexColorKey="color", vertexBorderColorKey="color")
p2 = Plotly.DataByGraph(g2, edgeWidth=2, vertexSizeKey="id", vertexColorKey="color", vertexBorderColorKey="color")
figure = Plotly.FigureByData(p1+p2)
Plotly.Show(figure)

### Test all the 12 (6x2) possible boolean operations (union, intersect, difference, symmetric difference, impose, and print)

In [ ]:

# Choose an option by number:
# 1 Union / Merge (A + B)
# 2 Union / Merge (B + A)
# 3 Intersect (A X B)
# 4 Intersect (B X A)
# 5 Difference (A - B)
# 6 Difference (B - A)
# 7 Symmetric Difference / XOR (A <> B)
# 8 Symmetric Difference / XOR (B <> A)
# 9 Impose (A <- B)
# 10 Impose (B <- A)
# 11 Imprint (A <- B)
# 12 Imprint (B <- A)

# SPECIFY OPTION NUMBER HERE and RE-RUN THIS CELL
option = 12

### Do not change code below this line unless you know what you are doing.
option = min(max(1, option), 12)
mapping = {1: "Union / Merge (A + B)",
           2: "Union / Merge (B + A)",
           3: "Intersect (A X B)",
           4: "Intersect (B X A)",
           5: "Difference (A - B)",
           6: "Difference (B - A)",
           7: "Symmetric Difference / XOR (A <> B)",
           8: "Symmetric Difference / XOR (B <> A)",
           9: "Impose (A <- B)",
           10: "Impose (B <- A)",
           11: "Imprint(A <- B)",
           12: "Imprint(B <-A)"}
print("You chose option", option,":",mapping[option])

if option == 1:
    result = TGraph.Union(g1, g2, vertexKeys=["id"])
elif option == 2:
    result = TGraph.Union(g2, g1, vertexKeys=["id"])
elif option == 3:
    result = TGraph.Intersect(g1, g2)
elif option == 4:
    result = TGraph.Intersect(g2, g1)
elif option == 5:
    result = TGraph.Difference(g1, g2)
elif option == 6:
    result = TGraph.Difference(g2, g1)
elif option == 7:
    result = TGraph.SymmetricDifference(g1, g2)
elif option == 8:
    result = TGraph.SymmetricDifference(g2, g1)
elif option == 9:
    result = TGraph.Impose(g1, g2)
elif option == 10:
    result = TGraph.Impose(g2, g1)
elif option == 11:
    result = TGraph.Imprint(g1, g2)
elif option == 12:
    result = TGraph.Imprint(g2, g1)

print(result)
if result:
    vertices = TGraph.Vertices(result)
    print("Vertices:")
    for v in vertices:
        d = _record_dictionary(v)
        if option == 1 or option == 2 or option == 3 or option == 4:
            # The color is now a list. We need to fix that
            color_value = Dictionary.ValueAtKey(d, "color")
            if isinstance(color_value, list):
                color_value = [Color.AnyToHex(c) for c in color_value]
                final_color = Color.AverageHex(color_value)
                d = Dictionary.SetValueAtKey(d, "color", final_color)
        print(" ", Dictionary.Keys(d), Dictionary.Values(d))
        v = Topology.SetDictionary(v, d)

    edges = _edge_records(result)
    print("Edges:")
    for e in edges:
        d = _record_dictionary(e)
        print(" ", Dictionary.Keys(d), Dictionary.Values(d))
    Topology.Show(result, vertexSizeKey="id", vertexColorKey="color")
else:
    print("No Graph was created.")